In [1]:
from pathlib import Path
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

In [2]:
ROOT = Path.cwd().parent

DATA_PATH = ROOT / "dataset" / "data" / "combined_train.csv"
CHECKPOINT_DIR = ROOT / "checkpoints"

CHECKPOINT_DIR.mkdir(exist_ok=True)

print("Project root:", ROOT)
print("Training data:", DATA_PATH)
print("Checkpoint directory:", CHECKPOINT_DIR)

Project root: c:\New folder\Projects\sentiment analysis
Training data: c:\New folder\Projects\sentiment analysis\dataset\data\combined_train.csv
Checkpoint directory: c:\New folder\Projects\sentiment analysis\checkpoints


In [3]:
data = pd.read_csv(DATA_PATH)

print("Original data shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

print("\nOriginal polarity values:")
print(data["polarity"].value_counts(dropna=False))

display(data.head())

Original data shape: (2734, 5)

Columns:
['Unnamed: 0.1', 'Unnamed: 0', 'text', 'aspect_term', 'polarity']

Original polarity values:
polarity
positive    1196
negative    1018
neutral      473
conflict      45
NaN            2
Name: count, dtype: int64


,Unnamed: 0.1,Unnamed: 0,text,aspect_term,polarity
0,0,0,lenovo personally maybe,lenovo,positive
1,1,1,legion like intel chipset great laptop value b...,intel,positive
2,2,2,legion like intel chipset great laptop value b...,battery life,positive
3,3,3,definitely consider lenovo gaming laptop,lenovo,positive
4,4,4,hp user not hp,hp,negative


In [ ]:
LABEL_FIXES = {
    "postive": "positive",
    "po": "positive",
}

VALID_LABELS = {
    "positive",
    "negative",
    "neutral",
    "conflict",
}

data = data.copy()

# Normalize polarity labels.
data["polarity"] = (
    data["polarity"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(LABEL_FIXES)
)

data = data.dropna(subset=["text", "aspect_term", "polarity"])

data = data[
    data["text"].astype(str).str.strip().ne("") &
    data["aspect_term"].astype(str).str.strip().ne("")
].copy()

data = data[data["polarity"].isin(VALID_LABELS)].copy()

data["input_text"] = (
    data["aspect_term"].astype(str)
    + " [SEP] "
    + data["text"].astype(str)
)

data = data.reset_index(drop=True)

print("Cleaned data shape:", data.shape)
print("\nCleaned class distribution:")
print(data["polarity"].value_counts())

display(data[["text", "aspect_term", "polarity", "input_text"]].head())

Cleaned data shape: (2721, 6)

Cleaned class distribution:
polarity
positive    1193
negative    1013
neutral      470
conflict      45
Name: count, dtype: int64


,text,aspect_term,polarity,input_text
0,lenovo personally maybe,lenovo,positive,lenovo [SEP] lenovo personally maybe
1,legion like intel chipset great laptop value b...,intel,positive,intel [SEP] legion like intel chipset great la...
2,legion like intel chipset great laptop value b...,battery life,positive,battery life [SEP] legion like intel chipset g...
3,definitely consider lenovo gaming laptop,lenovo,positive,lenovo [SEP] definitely consider lenovo gaming...
4,hp user not hp,hp,negative,hp [SEP] hp user not hp


In [5]:
X = data["input_text"]
y = data["polarity"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training examples:", len(X_train_text))
print("Testing examples:", len(X_test_text))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training examples: 2176
Testing examples: 545

Training label distribution:
polarity
positive    954
negative    810
neutral     376
conflict     36
Name: count, dtype: int64

Testing label distribution:
polarity
positive    239
negative    203
neutral      94
conflict      9
Name: count, dtype: int64


In [6]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words="english"
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("X_train type:", type(X_train))
print("X_train shape:", X_train.shape)

print("X_test type:", type(X_test))
print("X_test shape:", X_test.shape)

print("Vocabulary size:", len(vectorizer.vocabulary_))

X_train type: <class 'scipy.sparse._csr.csr_matrix'>
X_train shape: (2176, 5000)
X_test type: <class 'scipy.sparse._csr.csr_matrix'>
X_test shape: (545, 5000)
Vocabulary size: 5000


In [7]:
models = {
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "linear_svc": LinearSVC(
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    )
}

results = {}
trained_models = {}

for name, model in models.items():
    print("=" * 70)
    print(f"Training: {name}")

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(
        y_test,
        predictions,
        average="macro",
        zero_division=0
    )

    results[name] = {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "predictions": predictions
    }

    trained_models[name] = model

    print("Accuracy:", round(accuracy, 4))
    print("Macro-F1:", round(macro_f1, 4))

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions))
    print()

Training: logistic_regression
Accuracy: 0.6422
Macro-F1: 0.4893

Classification report:
              precision    recall  f1-score   support

    conflict       0.06      0.11      0.08         9
    negative       0.67      0.67      0.67       203
     neutral       0.46      0.53      0.50        94
    positive       0.76      0.68      0.72       239

    accuracy                           0.64       545
   macro avg       0.49      0.50      0.49       545
weighted avg       0.66      0.64      0.65       545

Confusion matrix:
[[  1   4   1   3]
 [  8 137  26  32]
 [  0  27  50  17]
 [  8  38  31 162]]

Training: linear_svc
Accuracy: 0.6972
Macro-F1: 0.506

Classification report:
              precision    recall  f1-score   support

    conflict       0.00      0.00      0.00         9
    negative       0.70      0.73      0.72       203
     neutral       0.52      0.56      0.54        94
    positive       0.78      0.74      0.76       239

    accuracy                   

In [8]:
comparison = pd.DataFrame(
    [
        {
            "model": model_name,
            "accuracy": result["accuracy"],
            "macro_f1": result["macro_f1"]
        }
        for model_name, result in results.items()
    ]
).sort_values("macro_f1", ascending=False)

display(comparison)

best_model_name = comparison.iloc[0]["model"]
best_model = trained_models[best_model_name]

print(
    f"Selected baseline model: {best_model_name} | "
    f"Macro-F1: {results[best_model_name]['macro_f1']:.4f}"
)

,model,accuracy,macro_f1
1,linear_svc,0.697248,0.505994
0,logistic_regression,0.642202,0.489283


Selected baseline model: linear_svc | Macro-F1: 0.5060


In [9]:
VECTORIZER_PATH = CHECKPOINT_DIR / "tfidf_vectorizer.joblib"
CLASSIFIER_PATH = CHECKPOINT_DIR / "tfidf_classifier.joblib"
MODEL_INFO_PATH = CHECKPOINT_DIR / "tfidf_model_info.joblib"

joblib.dump(vectorizer, VECTORIZER_PATH)
joblib.dump(best_model, CLASSIFIER_PATH)

model_info = {
    "model_name": best_model_name,
    "classes": list(best_model.classes_),
    "accuracy": results[best_model_name]["accuracy"],
    "macro_f1": results[best_model_name]["macro_f1"],
    "input_format": "aspect_term [SEP] text"
}

joblib.dump(model_info, MODEL_INFO_PATH)

print("Saved:")
print(VECTORIZER_PATH)
print(CLASSIFIER_PATH)
print(MODEL_INFO_PATH)

print("\nSaved model information:")
print(model_info)

Saved:
c:\New folder\Projects\sentiment analysis\checkpoints\tfidf_vectorizer.joblib
c:\New folder\Projects\sentiment analysis\checkpoints\tfidf_classifier.joblib
c:\New folder\Projects\sentiment analysis\checkpoints\tfidf_model_info.joblib

Saved model information:
{'model_name': 'linear_svc', 'classes': ['conflict', 'negative', 'neutral', 'positive'], 'accuracy': 0.6972477064220184, 'macro_f1': 0.5059936666448366, 'input_format': 'aspect_term [SEP] text'}


In [ ]:
loaded_vectorizer = joblib.load(VECTORIZER_PATH)
loaded_classifier = joblib.load(CLASSIFIER_PATH)

def predict_tfidf_polarity(text, aspect_term):
    model_input = f"{aspect_term} [SEP] {text}"
    vector = loaded_vectorizer.transform([model_input])

    prediction = loaded_classifier.predict(vector)[0]

    output = {
        "text": text,
        "aspect_term": aspect_term,
        "predicted_polarity": prediction
    }

    if hasattr(loaded_classifier, "predict_proba"):
        probabilities = loaded_classifier.predict_proba(vector)[0]

        output["confidence"] = round(float(probabilities.max()), 4)
        output["all_probabilities"] = {
            class_name: round(float(probability), 4)
            for class_name, probability in zip(
                loaded_classifier.classes_,
                probabilities
            )
        }

    return output

In [11]:
sample_comment = """
The laptop is excellent for school, browsing, coding, and Office work.
However, the 8GB RAM is restrictive and AAA gaming will struggle.
"""

aspects_to_test = [
    "school",
    "productivity",
    "8gb ram",
    "gaming performance"
]

for aspect in aspects_to_test:
    print(predict_tfidf_polarity(sample_comment, aspect))

{'text': '\nThe laptop is excellent for school, browsing, coding, and Office work.\nHowever, the 8GB RAM is restrictive and AAA gaming will struggle.\n', 'aspect_term': 'school', 'predicted_polarity': 'positive'}
{'text': '\nThe laptop is excellent for school, browsing, coding, and Office work.\nHowever, the 8GB RAM is restrictive and AAA gaming will struggle.\n', 'aspect_term': 'productivity', 'predicted_polarity': 'positive'}
{'text': '\nThe laptop is excellent for school, browsing, coding, and Office work.\nHowever, the 8GB RAM is restrictive and AAA gaming will struggle.\n', 'aspect_term': '8gb ram', 'predicted_polarity': 'negative'}
{'text': '\nThe laptop is excellent for school, browsing, coding, and Office work.\nHowever, the 8GB RAM is restrictive and AAA gaming will struggle.\n', 'aspect_term': 'gaming performance', 'predicted_polarity': 'negative'}
